# 03 — Modality Shapley (Phase 1, step 1)
How much did the **tabular** branch (T) and the **image** branch (I) each contribute to the fused CWF prediction?

Two-player cooperative game, solved **exactly** (4 fusion evaluations per case):
- payoff `f(S)` = fused probability of the **predicted** class, with an absent modality replaced by a baseline vector (primary: training class prior; robustness: uniform)
- `phi_T = 1/2[f(T,I)-f(b,I)] + 1/2[f(T,b)-f(b,b)]`, `phi_I` symmetric, and `phi_T + phi_I = f(T,I) - f(b,b)` (checked numerically)
- tabular share `rho_T = |phi_T| / (|phi_T| + |phi_I|)`

Also: a model-level version (payoff = macro-F1 / per-class recall), disagreement analysis, error analysis.
**OOF rows use cross-fitted CWF weights (as in 02); test rows use weights fit on all OOF. Nothing is retrained and the test set is only *explained*, not used to choose anything.**

In [1]:
import os
from pathlib import Path
ROOT = Path(r"D:\skin_project"); OUT_DIR = ROOT/"outputs"
import json
import numpy as np, pandas as pd
from scipy.stats import mannwhitneyu, spearmanr
from sklearn.metrics import recall_score
from common import META

d = np.load(OUT_DIR/"base_outputs_v1.npz", allow_pickle=True)
CLASSES = [str(c) for c in d["classes"]]; C = len(CLASSES)
oof_T, oof_I, te_T, te_I = d["oof_xgb"], d["oof_vit"], d["test_xgb"], d["test_vit"]
y_tr, y_te, fold = d["y_train"], d["y_test"], d["fold"]

sp = json.load(open(OUT_DIR/"split_v1.json"))
meta = pd.read_csv(META); id2pat = dict(zip(meta.img_id, meta.patient_id.astype(str)))
pat_tr = np.array([id2pat[i] for i in sp["train_img_ids"]]); pat_te = np.array([id2pat[i] for i in sp["test_img_ids"]])
assert len(pat_tr) == len(y_tr) and len(pat_te) == len(y_te)
print(CLASSES, "| train", len(y_tr), "| test", len(y_te))

['ACK', 'BCC', 'MEL', 'NEV', 'SCC', 'SEK'] | train 1837 | test 461


## 3a. CWF + exact two-player Shapley

In [2]:
def macro_f1(y, p):
    cm = np.bincount(y*C + p, minlength=C*C).reshape(C, C); tp = np.diag(cm).astype(float)
    prec = tp/np.maximum(cm.sum(0), 1); rec = tp/np.maximum(cm.sum(1), 1)
    return float(np.mean(2*prec*rec/np.maximum(prec+rec, 1e-12)))

def cwf_weights(PT, PI, y):
    rT = recall_score(y, PT.argmax(1), average=None, labels=range(C), zero_division=0)
    rI = recall_score(y, PI.argmax(1), average=None, labels=range(C), zero_division=0)
    return np.where(rT + rI > 0, rT/np.maximum(rT + rI, 1e-12), 0.5)

def cwf_apply(PT, PI, wT):
    s = wT*PT + (1 - wT)*PI
    return s/s.sum(1, keepdims=True)

def shapley_two(PT, PI, wT, bT, bI):
    """exact Shapley for players {T, I}; absent player -> baseline vector. Returns dict of per-case arrays."""
    BT = np.broadcast_to(bT, PT.shape); BI = np.broadcast_to(bI, PI.shape)
    F = {"TI": cwf_apply(PT, PI, wT), "bI": cwf_apply(BT, PI, wT), "Tb": cwf_apply(PT, BI, wT), "bb": cwf_apply(BT, BI, wT)}
    c = F["TI"].argmax(1); r = np.arange(len(c)); f = {k: v[r, c] for k, v in F.items()}
    phiT = .5*(f["TI"] - f["bI"]) + .5*(f["Tb"] - f["bb"])
    phiI = .5*(f["TI"] - f["Tb"]) + .5*(f["bI"] - f["bb"])
    assert np.allclose(phiT + phiI, f["TI"] - f["bb"]), "efficiency violated"
    rho = np.abs(phiT)/np.maximum(np.abs(phiT) + np.abs(phiI), 1e-12)
    return dict(pred=c, phiT=phiT, phiI=phiI, rho=rho, f=f["TI"], base=f["bb"])

prior = np.bincount(y_tr, minlength=C)/len(y_tr); uniform = np.full(C, 1/C)
wT_full = cwf_weights(oof_T, oof_I, y_tr)

def run_oof(bvec):
    out = {k: np.zeros(len(y_tr)) for k in ("phiT", "phiI", "rho", "f", "base")}; out["pred"] = np.zeros(len(y_tr), int)
    for k in range(5):
        tr, va = fold != k, fold == k
        r = shapley_two(oof_T[va], oof_I[va], cwf_weights(oof_T[tr], oof_I[tr], y_tr[tr]), bvec, bvec)
        for key in out: out[key][va] = r[key]
    return out

S_oof = run_oof(prior); S_te = shapley_two(te_T, te_I, wT_full, prior, prior)
S_oof_u = run_oof(uniform); S_te_u = shapley_two(te_T, te_I, wT_full, uniform, uniform)

fp = OUT_DIR/"fusion_v1.npz"
if fp.exists():
    fz = np.load(fp, allow_pickle=True)
    print("CWF weights match fusion_v1.npz:", np.allclose(fz["w_xgb"], wT_full), "| selected:", fz["selected"])
print("efficiency check passed. mean f(T,I)=%.3f, mean f(prior,prior)=%.3f" % (S_oof["f"].mean(), S_oof["base"].mean()))
print("OOF fused macro-F1 (cross-fitted): %.3f" % macro_f1(y_tr, S_oof["pred"]))

CWF weights match fusion_v1.npz: True | selected: CWF
efficiency check passed. mean f(T,I)=0.656, mean f(prior,prior)=0.277
OOF fused macro-F1 (cross-fitted): 0.634


## 3b. Tabular share per class (OOF = all 1,837 training rows; test = 461) with patient-cluster bootstrap CIs

In [3]:
def boot_mean_by_class(x, y, pat, B=1000, seed=0):
    rng = np.random.default_rng(seed); u, inv = np.unique(pat, return_inverse=True)
    groups = [np.where(inv == i)[0] for i in range(len(u))]; M = np.full((B, C), np.nan)
    for b in range(B):
        idx = np.concatenate([groups[i] for i in rng.integers(0, len(u), len(u))])
        xb, yb = x[idx], y[idx]
        for c in range(C):
            m = yb == c
            if m.any(): M[b, c] = xb[m].mean()
    return np.nanpercentile(M, [2.5, 97.5], axis=0)

def table(S, y, pat, label):
    lo, hi = boot_mean_by_class(S["rho"], y, pat)
    rows = []
    for c in range(C):
        m = y == c
        rows.append(dict(true_class=CLASSES[c], n=int(m.sum()), mean_rhoT=S["rho"][m].mean(), CI=(round(lo[c], 3), round(hi[c], 3)),
                         median_rhoT=np.median(S["rho"][m]), pct_T_dominant=(S["rho"][m] > .5).mean(),
                         pct_T_negative=(S["phiT"][m] < 0).mean(), pct_I_negative=(S["phiI"][m] < 0).mean()))
    rows.append(dict(true_class="ALL", n=len(y), mean_rhoT=S["rho"].mean(), CI=("", ""), median_rhoT=np.median(S["rho"]),
                     pct_T_dominant=(S["rho"] > .5).mean(), pct_T_negative=(S["phiT"] < 0).mean(), pct_I_negative=(S["phiI"] < 0).mean()))
    print(f"--- {label}"); print(pd.DataFrame(rows).set_index("true_class").round(3).to_string()); print()

table(S_oof, y_tr, pat_tr, "OOF (cross-fitted weights), prior baseline")
table(S_te, y_te, pat_te, "TEST, prior baseline  (MEL n is tiny -> interpret with care)")

--- OOF (cross-fitted weights), prior baseline
               n  mean_rhoT              CI  median_rhoT  pct_T_dominant  pct_T_negative  pct_I_negative
true_class                                                                                              
ACK          603      0.607   (0.59, 0.624)        0.602           0.789           0.053           0.217
BCC          671      0.662  (0.641, 0.682)        0.690           0.815           0.066           0.301
MEL           45      0.374  (0.298, 0.457)        0.418           0.356           0.178           0.222
NEV          195      0.553  (0.518, 0.588)        0.545           0.718           0.056           0.118
SCC          143      0.653  (0.613, 0.693)        0.699           0.776           0.119           0.364
SEK          180      0.470   (0.429, 0.51)        0.490           0.478           0.128           0.244
ALL         1837      0.606            (, )        0.624           0.749           0.073           0.251

--- TES

## 3c. Does modality reliance differ for correct vs wrong predictions?

In [4]:
for label, S, y in (("OOF", S_oof, y_tr), ("TEST", S_te, y_te)):
    ok = S["pred"] == y
    u_, p_ = mannwhitneyu(S["rho"][ok], S["rho"][~ok])
    print(f"{label}: mean rho_T correct {S['rho'][ok].mean():.3f} (n={ok.sum()}) vs wrong {S['rho'][~ok].mean():.3f} (n={(~ok).sum()}) | Mann-Whitney p = {p_:.3g}")
    print("   mean phi_T / phi_I  correct: %.3f / %.3f   wrong: %.3f / %.3f" % (
        S["phiT"][ok].mean(), S["phiI"][ok].mean(), S["phiT"][~ok].mean(), S["phiI"][~ok].mean()))

OOF: mean rho_T correct 0.598 (n=1361) vs wrong 0.628 (n=476) | Mann-Whitney p = 0.000432
   mean phi_T / phi_I  correct: 0.257 / 0.161   wrong: 0.220 / 0.051
TEST: mean rho_T correct 0.600 (n=344) vs wrong 0.643 (n=117) | Mann-Whitney p = 0.0398
   mean phi_T / phi_I  correct: 0.248 / 0.159   wrong: 0.220 / 0.060


## 3d. Disagreement analysis (do the branches agree on top-1? whom did the fusion follow?)

In [5]:
def disagreement(PT, PI, S, y, label):
    pt, pi, pf = PT.argmax(1), PI.argmax(1), S["pred"]; agree = pt == pi
    print(f"--- {label}: branches agree on {agree.mean():.1%} of cases ({agree.sum()}/{len(y)})")
    print("  accuracy | agree: fused %.3f | disagree: fused %.3f, XGB %.3f, ViT %.3f (n=%d)" % (
        (pf == y)[agree].mean(), (pf == y)[~agree].mean(), (pt == y)[~agree].mean(), (pi == y)[~agree].mean(), (~agree).sum()))
    dg = ~agree
    print("  when they disagree, fusion followed: XGB %.1f%% | ViT %.1f%% | neither %.1f%%" % (
        100*(pf[dg] == pt[dg]).mean(), 100*(pf[dg] == pi[dg]).mean(), 100*((pf[dg] != pt[dg]) & (pf[dg] != pi[dg])).mean()))
    fx = pf[dg] == pt[dg]
    print("  accuracy of the branch followed: XGB-followed %.3f (n=%d) | ViT-followed %.3f (n=%d)" % (
        (pf == y)[dg][fx].mean() if fx.any() else np.nan, fx.sum(), (pf == y)[dg][~fx].mean() if (~fx).any() else np.nan, (~fx).sum()))
    print("  mean rho_T: agree %.3f | disagree %.3f" % (S["rho"][agree].mean(), S["rho"][dg].mean()))
    pairs = pd.Series([f"XGB={CLASSES[a]} vs ViT={CLASSES[b]}" for a, b in zip(pt[dg], pi[dg])]).value_counts().head(8)
    print("  most frequent conflicts:\n" + pairs.to_string() + "\n")

disagreement(oof_T, oof_I, S_oof, y_tr, "OOF")
disagreement(te_T, te_I, S_te, y_te, "TEST")

--- OOF: branches agree on 54.5% of cases (1002/1837)
  accuracy | agree: fused 0.879 | disagree: fused 0.575, XGB 0.404, ViT 0.390 (n=835)
  when they disagree, fusion followed: XGB 68.5% | ViT 29.8% | neither 1.7%
  accuracy of the branch followed: XGB-followed 0.528 (n=572) | ViT-followed 0.677 (n=263)
  mean rho_T: agree 0.609 | disagree 0.602
  most frequent conflicts:
XGB=ACK vs ViT=BCC    115
XGB=BCC vs ViT=ACK    108
XGB=BCC vs ViT=SCC     97
XGB=BCC vs ViT=SEK     62
XGB=ACK vs ViT=SCC     58
XGB=ACK vs ViT=SEK     42
XGB=BCC vs ViT=NEV     37
XGB=SCC vs ViT=ACK     36

--- TEST: branches agree on 55.3% of cases (255/461)
  accuracy | agree: fused 0.843 | disagree: fused 0.626, XGB 0.432, ViT 0.403 (n=206)
  when they disagree, fusion followed: XGB 66.0% | ViT 31.6% | neither 2.4%
  accuracy of the branch followed: XGB-followed 0.588 (n=136) | ViT-followed 0.700 (n=70)
  mean rho_T: agree 0.609 | disagree 0.613
  most frequent conflicts:
XGB=BCC vs ViT=ACK    36
XGB=BCC vs ViT

## 3e. Robustness: uniform baseline instead of class prior

In [6]:
for label, A, B_ in (("OOF", S_oof, S_oof_u), ("TEST", S_te, S_te_u)):
    rs = spearmanr(A["rho"], B_["rho"]).statistic
    print(f"{label}: mean rho_T prior {A['rho'].mean():.3f} vs uniform {B_['rho'].mean():.3f} | Spearman(rho_prior, rho_uniform) = {rs:.3f} "
          f"| sign(phi_T) agreement {(np.sign(A['phiT']) == np.sign(B_['phiT'])).mean():.3f}")
table(S_oof_u, y_tr, pat_tr, "OOF, UNIFORM baseline (compare with 3b)")

OOF: mean rho_T prior 0.606 vs uniform 0.600 | Spearman(rho_prior, rho_uniform) = 0.843 | sign(phi_T) agreement 0.964
TEST: mean rho_T prior 0.611 vs uniform 0.598 | Spearman(rho_prior, rho_uniform) = 0.833 | sign(phi_T) agreement 0.939
--- OOF, UNIFORM baseline (compare with 3b)
               n  mean_rhoT              CI  median_rhoT  pct_T_dominant  pct_T_negative  pct_I_negative
true_class                                                                                              
ACK          603      0.598   (0.58, 0.615)        0.565           0.693           0.025           0.106
BCC          671      0.641  (0.622, 0.661)        0.645           0.790           0.036           0.130
MEL           45      0.403  (0.309, 0.513)        0.352           0.400           0.311           0.200
NEV          195      0.556   (0.52, 0.593)        0.536           0.590           0.077           0.118
SCC          143      0.655  (0.604, 0.702)        0.718           0.762           0.084 

## 3f. Model-level modality Shapley (payoff = macro-F1, and per-class recall)
Same 2-player game, but the payoff is a *dataset-level* score: absent modality is replaced by the baseline for every row. Bootstrap = patient clusters.

**Caveat for the per-class recall table:** with the class-prior baseline, the 'all-absent' model always predicts the majority class (BCC), so BCC recall is 1.0 at baseline and its phi values come out negative. That is an artefact of the baseline, not a finding. Read the macro-F1 game and the per-case rho_T first; treat per-class recall phis as secondary.

In [7]:
def hard_combos(PT, PI, wT, b):
    BT = np.broadcast_to(b, PT.shape); BI = np.broadcast_to(b, PI.shape)
    return {k: cwf_apply(*args, wT).argmax(1) for k, args in {"TI": (PT, PI), "bI": (BT, PI), "Tb": (PT, BI), "bb": (BT, BI)}.items()}

def model_level(PT, PI, wT, y, pat, label, B=1000):
    h = hard_combos(PT, PI, wT, prior)
    def phis(idx, fn):
        v = {k: fn(y[idx], p[idx]) for k, p in h.items()}
        return .5*(v["TI"] - v["bI"]) + .5*(v["Tb"] - v["bb"]), .5*(v["TI"] - v["Tb"]) + .5*(v["bI"] - v["bb"]), v
    allidx = np.arange(len(y)); pT_, pI_, v = phis(allidx, macro_f1)
    rng = np.random.default_rng(0); u, inv = np.unique(pat, return_inverse=True); groups = [np.where(inv == i)[0] for i in range(len(u))]
    bt, bi = np.empty(B), np.empty(B)
    for b in range(B):
        idx = np.concatenate([groups[i] for i in rng.integers(0, len(u), len(u))]); bt[b], bi[b], _ = phis(idx, macro_f1)
    print(f"--- {label}: macro-F1 of fused model {v['TI']:.3f}, prior-only {v['bb']:.3f}")
    print(f"  phi_T = {pT_:+.3f} {tuple(np.percentile(bt, [2.5, 97.5]).round(3))} | phi_I = {pI_:+.3f} {tuple(np.percentile(bi, [2.5, 97.5]).round(3))} | tabular share {pT_/(pT_+pI_):.2f}")
    rec = lambda yy, pp: recall_score(yy, pp, average=None, labels=range(C), zero_division=0)
    rows = {}
    for c in range(C):
        f = {k: rec(y, p)[c] for k, p in h.items()}
        t_ = .5*(f["TI"] - f["bI"]) + .5*(f["Tb"] - f["bb"]); i_ = .5*(f["TI"] - f["Tb"]) + .5*(f["bI"] - f["bb"])
        rows[CLASSES[c]] = dict(n=int((y == c).sum()), recall_fused=f["TI"], phi_T=t_, phi_I=i_)
    print("  per-class recall game:"); print(pd.DataFrame(rows).T.round(3).to_string()); print()

model_level(oof_T, oof_I, wT_full, y_tr, pat_tr, "OOF (weights fit on all OOF, so slightly optimistic)")
model_level(te_T, te_I, wT_full, y_te, pat_te, "TEST")

--- OOF (weights fit on all OOF, so slightly optimistic): macro-F1 of fused model 0.642, prior-only 0.089
  phi_T = +0.243 (np.float64(0.211), np.float64(0.275)) | phi_I = +0.310 (np.float64(0.278), np.float64(0.34)) | tabular share 0.44
  per-class recall game:
         n  recall_fused  phi_T  phi_I
ACK  603.0         0.846  0.447  0.399
BCC  671.0         0.784 -0.107 -0.109
MEL   45.0         0.489  0.111  0.378
NEV  195.0         0.810  0.428  0.382
SCC  143.0         0.245  0.140  0.105
SEK  180.0         0.633  0.161  0.472

--- TEST: macro-F1 of fused model 0.707, prior-only 0.091
  phi_T = +0.231 (np.float64(0.172), np.float64(0.297)) | phi_I = +0.385 (np.float64(0.316), np.float64(0.436)) | tabular share 0.37
  per-class recall game:
         n  recall_fused  phi_T  phi_I
ACK  127.0         0.827  0.429  0.398
BCC  174.0         0.799 -0.060 -0.141
MEL    7.0         1.000  0.214  0.786
NEV   49.0         0.878  0.520  0.357
SCC   49.0         0.184  0.020  0.163
SEK   55.0   

## 3g. Save for later steps (H-MSFI, OHA, explanation cards)

In [8]:
np.savez(OUT_DIR/"modality_shapley_v1.npz", classes=np.array(CLASSES), prior=prior, w_xgb=wT_full,
    oof_phiT=S_oof["phiT"], oof_phiI=S_oof["phiI"], oof_rho=S_oof["rho"], oof_pred=S_oof["pred"], oof_f=S_oof["f"], oof_base=S_oof["base"],
    test_phiT=S_te["phiT"], test_phiI=S_te["phiI"], test_rho=S_te["rho"], test_pred=S_te["pred"], test_f=S_te["f"], test_base=S_te["base"],
    oofU_rho=S_oof_u["rho"], testU_rho=S_te_u["rho"])
print("saved modality_shapley_v1.npz")

saved modality_shapley_v1.npz
